# Phase 2: Model Evaluation & Error Analysis

This notebook loads the trained classification artifacts, processes the test dataset partition, compares metrics against a baseline classifier, and performs error analysis across volatility regimes.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join("..")))

import pickle
import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import src.config as config
from src.features.indicators import compute_indicators, compute_stationary_features
from src.models.train import split_data_chronologically, prepare_features_and_targets
from src.models.evaluation import calculate_metrics, plot_confusion_matrix, plot_roc_curves

# Load saved model artifacts
artifacts_path = config.PROJECT_ROOT / "models" / "ml_artifacts.pkl"
print(f"Loading artifacts from: {artifacts_path}")
with open(artifacts_path, "rb") as f:
    artifacts = pickle.load(f)

scaler = artifacts["scaler"]
lr = artifacts["logistic_regression"]
rf = artifacts["random_forest"]
feature_cols = artifacts["feature_names"]

## 1. Load and Prepare Test Partition

In [ ]:
df = pl.read_parquet(config.SAMPLE_PARQUET_PATH).filter(pl.col("symbol") == config.TARGET_SYMBOL)
df_indicators = compute_indicators(df)
df_features = compute_stationary_features(df_indicators)

df_labeled = df_features.with_columns([
    (pl.col("close").shift(-config.FUTURE_HORIZON).over("symbol") / pl.col("close") - 1.0)
    .alias("future_return")
]).with_columns([
    pl.when(pl.col("future_return") > config.TARGET_THRESHOLD)
    .then(1)
    .otherwise(0)
    .alias("target")
]).drop_nulls()

# Get chronological test set
_, _, test_df = split_data_chronologically(
    df_labeled, train_end="2025-01-01", val_end="2025-07-01"
)

X_test, y_test = prepare_features_and_targets(test_df, feature_cols, "target")
X_test_scaled = scaler.transform(X_test)
print(f"Test features shape: {X_test.shape}")

## 2. Metrics Evaluation

In [ ]:
# Logistic Regression predictions
lr_preds = lr.predict(X_test_scaled)
lr_probs = lr.predict_proba(X_test_scaled)[:, 1]

# Random Forest predictions
rf_preds = rf.predict(X_test_scaled)
rf_probs = rf.predict_proba(X_test_scaled)[:, 1]

# Baseline model: Majority class predict (predicts 1 everywhere)
baseline_preds = np.ones_like(y_test)
baseline_probs = np.ones_like(y_test) * 0.5

# Compute metrics
lr_metrics = calculate_metrics(y_test, lr_preds, lr_probs)
rf_metrics = calculate_metrics(y_test, rf_preds, rf_probs)
baseline_metrics = calculate_metrics(y_test, baseline_preds, baseline_probs)

# Format as DataFrame
results_df = pd.DataFrame({
    "Baseline (Majority Class)": baseline_metrics,
    "Logistic Regression": lr_metrics,
    "Random Forest": rf_metrics
}).T

print("Performance comparison on test partition:")
results_df

## 3. Comparative Visualizations

In [ ]:
# Plot Confusion Matrices
plot_confusion_matrix(y_test, lr_preds, title="Logistic Regression Confusion Matrix")
plt.show()

plot_confusion_matrix(y_test, rf_preds, title="Random Forest Confusion Matrix")
plt.show()

# Plot comparative ROC curves
plot_roc_curves({
    "Logistic Regression": lr_probs,
    "Random Forest": rf_probs
}, y_test)
plt.show()

## 4. Volatility Regime Error Analysis
Financial ML models often behave differently during quiet vs. highly volatile market periods. We split accuracy based on the median historical volatility to identify performance biases.

In [ ]:
analysis_df = test_df.to_pandas()
analysis_df["lr_correct"] = (lr_preds == y_test).astype(int)
analysis_df["rf_correct"] = (rf_preds == y_test).astype(int)

# Split by volatility threshold
median_vol = analysis_df["volatility_30"].median()
low_vol_df = analysis_df[analysis_df["volatility_30"] <= median_vol]
high_vol_df = analysis_df[analysis_df["volatility_30"] > median_vol]

print(f"Median Volatility: {median_vol:.6f}")
print(f"Logistic Regression accuracy - Low Volatility: {low_vol_df['lr_correct'].mean():.4f}")
print(f"Logistic Regression accuracy - High Volatility: {high_vol_df['lr_correct'].mean():.4f}")

print(f"\nRandom Forest accuracy - Low Volatility: {low_vol_df['rf_correct'].mean():.4f}")
print(f"Random Forest accuracy - High Volatility: {high_vol_df['rf_correct'].mean():.4f}")